# Linear Autoregression and Recursive Forecasting


In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(7)
t = np.arange(400)
series = 0.05 * t + 3 * np.sin(2 * np.pi * t / 20) + rng.randn(400) * 0.3

def make_windows(series, lookback):
    series = np.asarray(series)
    N = len(series) - lookback
    X = np.array([series[i:i + lookback] for i in range(N)])
    y = np.array([series[i + lookback] for i in range(N)])
    return X, y

def fit_ar_forecaster(X_train, y_train):
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model

def forecast_step(model, window):
    window = np.asarray(window).reshape(1, -1)
    return float(model.predict(window)[0])

def recursive_forecast(model, seed_window, steps):
    window = np.asarray(seed_window, dtype=float).copy()
    predictions = []
    for _ in range(steps):
        pred = forecast_step(model, window)
        predictions.append(pred)
        window = np.append(window[1:], pred)
    return np.array(predictions)

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# Verification
s = 3 * np.arange(40) + 5.0
X, y = make_windows(s, 6)
model = fit_ar_forecaster(X, y)
print(round(forecast_step(model, s[:6]), 3))
preds = recursive_forecast(model, s[:6], 10)
print(np.allclose(preds, s[6:16], atol=1e-3))
print(round(rmse([1, 2, 3], [1, 2, 5]), 4))


23.0
True
1.1547
